# Security Demo — Separation of Duties, REVOKE + Audit Log

Topic 5 — Separation of Duties, Instant REVOKE, Audit

**Part 1 — Separation of Duties:** The Business Analyst has their own sandbox schema. They cannot touch governed SILVER tables.

**Part 2 — Instant REVOKE:** HIPAA requires access can be removed immediately — zero lag, no session invalidation.

**Part 3 — Audit Log:** Snowflake captures every query, including 0-row results from row policies and the 'not authorized' error from Part 2.

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;
USE SECONDARY ROLES NONE;
USE DATABASE zFACETS_DEV_CLONE;
USE SCHEMA SILVER;
USE WAREHOUSE WH_XS;

## Part 1 — Separation of Duties

The `BUSINESS_ANALYST_ROLE` owns their `ANALYST` sandbox schema but **cannot modify or drop governed SILVER tables**. Row access policies travel with the data — a CTAS into the BA's sandbox only contains COMM rows.

In [ ]:
%%sql
USE ROLE BUSINESS_ANALYST_ROLE;

-- BA creates their own sandbox schema
CREATE SCHEMA IF NOT EXISTS ANALYST
    COMMENT = 'Business Analyst sandbox — read/write here, read-only on SILVER.';

-- CTAS: row access policy enforces COMM-only rows even in the BA sandbox
CREATE OR REPLACE TABLE ANALYST.COMM_MEMBERS AS
SELECT MEME_ID, MEME_MCTR_TYPE, MEME_REL_CD, RELATIONSHIP_DESC, MEMBER_STATUS, ACTIVE_PCP_NAME
FROM SILVER.MEMBER;

SELECT * FROM ANALYST.COMM_MEMBERS LIMIT 10;
-- Note: only COMM rows — row access policy followed the data into the CTAS

In [ ]:
%%sql
-- BA creates a view — works fine in their sandbox
CREATE OR REPLACE VIEW ANALYST.COMM_ACTIVE_MEMBERS AS
SELECT MEME_ID, MEME_MCTR_TYPE, MEME_REL_CD, MEMBER_STATUS
FROM SILVER.MEMBER
WHERE MEMBER_STATUS = 'Active';

SELECT COUNT(*) AS active_commercial_members FROM ANALYST.COMM_ACTIVE_MEMBERS;

### Attempting to break out of the sandbox

`SELECT` was explicitly granted on `SILVER.MEMBER` — `INSERT` and DDL were not. Both fail with insufficient privileges.

In [ ]:
%%sql
-- Expected: Insufficient privileges to INSERT
INSERT INTO SILVER.MEMBER (MEME_ID, SBSB_ID, MEME_LAST_NAME, MEME_FIRST_NAME, MEME_MCTR_TYPE)
VALUES (99999, 99999, 'TEST', 'RECORD', 'COMM');

In [ ]:
%%sql
-- Expected: Insufficient privileges to DROP
DROP TABLE SILVER.MEMBER;

## Part 2 — Instant REVOKE

HIPAA requires access can be removed immediately. We revoke all three levels (table → schema → database) and verify it takes effect on the very next query — zero lag, no session restart.

In [ ]:
%%sql
-- Confirm BA currently has access (~30,593 COMM-only rows via row policy)
USE ROLE BUSINESS_ANALYST_ROLE;
SELECT COUNT(*) AS visible_members FROM SILVER.MEMBER;

In [ ]:
%%sql
USE ROLE ACCOUNTADMIN;

REVOKE SELECT ON TABLE zFACETS_DEV_CLONE.SILVER.MEMBER FROM ROLE BUSINESS_ANALYST_ROLE;
REVOKE USAGE ON SCHEMA zFACETS_DEV_CLONE.SILVER         FROM ROLE BUSINESS_ANALYST_ROLE;
REVOKE USAGE ON DATABASE zFACETS_DEV_CLONE              FROM ROLE BUSINESS_ANALYST_ROLE;

SHOW GRANTS TO ROLE BUSINESS_ANALYST_ROLE;

In [ ]:
%%sql
-- Expected: Database 'ZFACETS_DEV_CLONE' does not exist or not authorized.
-- Zero lag — enforced on next query, no session restart needed
USE ROLE BUSINESS_ANALYST_ROLE;
SELECT COUNT(*) AS visible_members FROM SILVER.MEMBER;

In [ ]:
--Restore Access 
USE ROLE ACCOUNTADMIN;

GRANT SELECT ON TABLE zFACETS_DEV_CLONE.SILVER.MEMBER TO ROLE BUSINESS_ANALYST_ROLE;
GRANT USAGE ON SCHEMA zFACETS_DEV_CLONE.SILVER         TO ROLE BUSINESS_ANALYST_ROLE;
GRANT USAGE ON DATABASE zFACETS_DEV_CLONE              TO ROLE BUSINESS_ANALYST_ROLE;

SHOW GRANTS TO ROLE BUSINESS_ANALYST_ROLE;

## Part 3 — Audit Log

Snowflake captures every query via `SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY`, including 0-row results from row policies and the 'not authorized' error from Part 2.
Pre-built as a 90-day snapshot in `GOVERNANCE_CA_DEMO.POLICY_STORE.ACCOUNT_ACCESS_HISTORY`.

In [ ]:
%%sql -r dataframe_9
USE ROLE ACCOUNTADMIN;
USE DATABASE GOVERNANCE_CA_DEMO;
USE SCHEMA POLICY_STORE;

-- No-op if already built by setup
CREATE TABLE IF NOT EXISTS ACCESS_HISTORY AS
SELECT
    ah.QUERY_ID, ah.QUERY_START_TIME,
    qh.USER_NAME, qh.ROLE_NAME,
    LEFT(qh.QUERY_TEXT, 200)                           AS query_preview,
    qh.EXECUTION_STATUS,
    ah.DIRECT_OBJECTS_ACCESSED[0]:objectName::STRING   AS first_object_accessed,
    ah.DIRECT_OBJECTS_ACCESSED[0]:objectDomain::STRING AS object_domain
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY ah
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY qh ON ah.QUERY_ID = qh.QUERY_ID
WHERE ah.QUERY_START_TIME >= DATEADD('day', -90, CURRENT_TIMESTAMP())
ORDER BY ah.QUERY_START_TIME DESC;

### Role-by-Role Access Summary

Who accessed what, how many times, and when — the view CalOptima auditors use for a HIPAA access review.

In [ ]:
%%sql -r dataframe_10
SELECT
    ROLE_NAME, USER_NAME,
    COUNT(*)              AS query_count,
    MIN(QUERY_START_TIME) AS first_access,
    MAX(QUERY_START_TIME) AS last_access
FROM ACCESS_HISTORY
WHERE ROLE_NAME IS NOT NULL 
GROUP BY ROLE_NAME, USER_NAME
ORDER BY query_count DESC;